# Feature selection - stage 2 depuis `greedy_kendall threshold=0.7`

Point de départ: les 34 features sorties de la phase corrélation (`greedy_kendall`, `threshold=0.7`).

Objectif: tester plusieurs familles de méthodes de feature selection, comparer les sous-ensembles avec le même protocole, puis générer un rapport HTML interactif.

## Méthodes incluses

- `base_greedy_kendall_corr_pruned`: référence avec les 34 features.
- `mi_topk`: information mutuelle feature-label.
- `anova_f_topk`: test F univarié.
- `univariate_auc_topk`: AUC univariée feature-label.
- `l1_logistic_topk`: coefficients non nuls/forts de logistic L1.
- `elasticnet_logistic_topk`: logistic ElasticNet.
- `random_forest_importance_topk`: importance Random Forest.
- `xgboost_importance_topk`: importance XGBoost.
- `permutation_importance_xgb_topk`: permutation importance avec modèle XGBoost.
- `shap_xgboost_topk`: importance SHAP XGBoost si `shap` est installé.
- `rfe_logistic_topk`: Recursive Feature Elimination.
- `boruta_like_xgb_topk`: comparaison avec features shadow.
- `mrmr_topk_penalty_05/10/20`: maximum relevance minimum redundancy.
- `forward_stepwise_xgb_inner_valid`: forward stepwise avec validation interne.
- `backward_stepwise_xgb_inner_valid`: backward stepwise avec validation interne.
- `family_quota_*`: variantes qui gardent syn/cls/tda.
- `rank_voting_ensemble_topk`: vote entre méthodes.
- `hybrid_mi_prefilter_mrmr`: préfiltre MI puis mRMR.

## Protocole d'évaluation

Pour chaque configuration évaluée: modèle entraîné sur le train de chaque fold, `threshold_opt_train` choisi sur le train, seuil appliqué au test, puis moyenne des métriques sur les 551 `pair_id`.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/Users/nahawandkired/Documents/metamatch')
SCRIPT = PROJECT_ROOT / 'scripts' / 'feature_selection_stage2_from_greedy_kendall34.py'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'exp_occidata' / 'reports' / 'meeting_baselines_vs_metamatch' / 'feature_selection_stage2_from_greedy_kendall34'

SCRIPT, OUT_DIR

## 1. Générer seulement le plan HTML, sans calcul F1

Cette cellule crée un rapport PLAN_ONLY rapide sans entraîner de modèles et sans calculer les rankers lourds.

In [ ]:
!python3 "$SCRIPT" \
  --output-root "$PROJECT_ROOT/outputs/exp_occidata" \
  --out-dir "$OUT_DIR" \
  --k-grid 5,8,10,12,15,20,25,30,34

## 2. Lancer l'évaluation complète et générer le vrai rapport HTML

Cette cellule peut être longue: elle entraîne les modèles et calcule F1/precision/recall sur les 551 paires de tables.

In [ ]:
!python3 "$SCRIPT" \
  --output-root "$PROJECT_ROOT/outputs/exp_occidata" \
  --out-dir "$OUT_DIR" \
  --sample-n 120000 \
  --eval-train-sample 0 \
  --k-grid 5,8,10,12,15,20,25,30,34 \
  --max-stepwise-k 15 \
  --run-evaluation

## Sorties attendues

- `candidate_feature_sets.csv`: toutes les configurations candidates.
- `feature_selection_stage2_summary.csv`: résumé global par méthode/config.
- `feature_selection_stage2_pair_id_evaluation.csv`: métriques par `pair_id`.
- `feature_selection_stage2_fold_evaluation.csv`: métriques par fold.
- `feature_selection_stage2_report.html`: rapport HTML final.